In [ ]:
import torch
import math

In [ ]:
def naive_softmax(x):

    """
    implementation of naive stable softmax
    """

    # substract max for numerical stability
    row_max = torch.max(x, axis=1, keepdims=True).values    # shape (n_rows, 1)
    x_stable = x - row_max

    # exponentiate
    x_exp = torch.exp(x_stable)

    # sum along rows
    row_sum = torch.sum(x_exp, axis=1, keepdims=True)

    # divide
    out = x_exp / row_sum

    return out

In [ ]:
def online_softmax(x):

    M, N = x.shape

    L_store = torch.zeros((M, ))

    for i in range(M):

        xi = x[i]

        m = float("-inf")
        d = 0

        for j in range(N):

            xj = xi[j].item()

            m_prev = m
            m = max(m, xj)

            d = d * math.exp(m_prev - m) + math.exp(xj - m)

        L_store[i] = m + math.log(d)

    return L_store


In [ ]:
def  online_blocked_softmax(x, block_size=64):
    """
    implementation of online softmax with block processing

    so instead of looping through element by element, we will
    instead loop through a block of elements at a time!
    """

    # get shape of x
    M, N = x.shape

    # create empty tensor to store final softmax output
    output = torch.empty_like(x)

    # we will parallelize over the row dimension
    for i in range(M):

        # index the row
        xi = x[i]

        # initialize for online softmax
        m = float("-inf")
        d = 0

        # loop over blocks of elements of data
        num_blocks = math.ceil(N / block_size)
        for j in range(num_blocks):

            # get start and end of block
            block_start = j * block_size
            block_end = block_start + block_size
            block_idx = torch.arange(block_start, block_end)

            # compute mask so we don't index invalid positions
            mask = (block_idx < N)
            block_idx = block_idx[mask]

            # load block
            block = xi[block_idx]

            # compute block max
            block_max = block.max().item()

            # update running max
            m_prev = m
            m = max(m, block_max)

            # update running denominator
            d = d * math.exp(m_prev - m) + torch.exp(block - m).sum().item()

        # compute softmax
        softmaxed_x = torch.exp(xi - m) / d

        # store in output tensor
        output[i] = softmaxed_x

    return output

In [ ]:
if __name__ == "__main__":

    rand = torch.randn(2, 8)
    naive_output = naive_softmax(rand)


In [ ]:
import torch
import math

In [ ]:
def flash_attention_forward(
        Q,
        K,
        V,
        softmax_scale=None,
        is_causal=False,
        BLOCK_SIZE=64,
        BLOCK_SIZE_KV=64
):

    # get the shape of the data we are working with
    batch_size, num_heads, seq_len, head_dim = Q.shape

    # use default softmax scaling if not provided
    if softmax_scale is None:
        softmax_scale = 1.0 / math.sqrt(head_dim)

    INV_LN2 = 1.442695040888963

    softmax_scale *= INV_LN2

    # prescale our Queries ([Q @ K.T] * softmax_scale * INV_LN2 is the same as [(Q * softmax_scale * INV_LN2) @ K.T])
    Q = Q * softmax_scale

    M = torch.full((batch_size, num_heads, seq_len), float("-inf"), device=Q.device, dtype=torch.float32)

    O = torch.zeros_like(Q)

    #triton parallel
    for b in range(batch_size):
        for h in range(num_heads):

            for q_start in range(0, seq_len, BLOCK_SIZE_Q):

                q_end = min(q_start + BLOCK_SIZE_Q, seq_len)

                q_indices = torch.arange(q_start, q_end, device=Q.device)

                Q_block = Q[b, h, q_start:q_end, :]     # block_q x head_dim

                m_i = torch.full((q_end - q_start, ), float("-inf"), device=Q.device, dtype=torch.float32)
                l_i = torch.zeros((q_end - q_start, ), device=Q.device, dtype=torch.float32)

                O_block = torch.zeros((q_end - q_start, head_dim), device=Q.device, dtype=torch.float32)

                if is_causal:

                    kv_start_prediag = 0
                    kv_end_prediag = q_start

                    kv_start_diag = q_start
                    kv_end_diag = q_end

                    kv_ranges = [
                        (kv_start_prediag, kv_end_prediag, "prediagonal"),
                        (kv_start_diag, kv_end_diag, "diag")
                    ]

                else:
                    kv_ranges = [(0, seq_len, "full")]

                for kv_start_range, kv_end_range, pass_type in kv_ranges:

                    for kv_start in range(kv_start_range, kv_end_range, BLOCK_SIZE_KV):

                        kv_end = min(kv_start + BLOCK_SIZE_KV, kv_end_range)

                        kv_indices = torch.arange(kv_start, kv_end, device=Q.device)

                        K_block = K[b, h, kv_start:kv_end, :]
                        V_block = V[b, h, kv_start:kv_end, :]

                        QK_BLOCK = Q_block @ K_block.T  # block_q x block_kv

                        if pass_type == "diag":

                            causal_mask = q_indices[:, None] > kv_indices[None, :]
                            QK_block = QK_BLOCK.masked_fill(~causal_mask, float("-inf"))

                        kv_padding_mask = (kv_indices < seq_len)
                        QK_block = QK_block.masked_fill(
                            ~kv_padding_mas[None, :], float("-inf")
                        )


                        m_ij = torch.maximum(m_i, QK_block.max(dim=1).values)
                        QK_block = QK_block - m_ij[:, None]
                        P_block = torch.exp2(QK_block)
                        l_ij = P_block.sum(dim=1)


                        alpha = torch.exp2(m_i - m_ij)
                        l_i = l_i * alpha + l_ij
                        O_block = O_block * alpha[:, None] + P_block @ V_block


                        m_i = m_ij


                m_i_logsumexp = m_i + torch.log2(l_i)

                M[b, h, q_start:q_end] = m_i_logsumexp

                O_block = O_block / (l_i[:, None] + 1e-6)

                O[b, h, q_start:q_end, :] = O_block

    return O, M


In [ ]:
if __name__ == "__main__":

    q = torch.randn(2, 4, 1024, 64, device="cuda", requires_grad=True)
    k = torch.randn(2, 4, 1024, 64, device="cuda", requires_grad=True)
    v = torch.randn(2, 4, 1024, 64, device="cuda", requires_grad=True)

    O, M = flash_attention_forward(q, k, v, is_causal=True)

    output = torch.nn.functional.scaled_dot_product_attention(
        q, k, v, is_causal=True
    )

In [ ]:
def flash_attention_backward(
        Q,      # (batch, num_heads, seq_len, head_dim)
        K,      # (batch, num_kv_heads, seq_len, head_dim)
        V,      # (batch, num_kv_heads, seq_len, head_dim)
        O,      # (batch, num_heads, seq_len, head_dim) - output from forward
        dO,     # (batch, num_heads, seq_len, head_dim) - gradient from upstream
        M,      # (batch, num_heads, seq_len) - logsumexp from forward pass
        softmax_scale=None,
        is_causal=False,
        BLOCK_SIZE_Q=64,
        BLOCK_SIZE_KV=64
):
    """
    backward pass for flash attention

    returns:
        dQ: gradient w.r.t Q
        dK: gradient w.r.t K
        dV: gradient w.r.t V
    """

    batch_size, num_heads, seq_len, head_dim = Q.shape

    # use default softmax scaling if not provided
    if softmax_scale is None:
        softmax_scale = 1.0 / math.sqrt(head_dim)

    # same adjustments as forward pass for exp2 vs exp
    INV_LN2 = 1.442695040888963
    LN2 = 0.693147182464        # 1 / INV_LN2 so we can unscale later

    soft_scale *= INV_LN2

    Q = Q * softmax_scale

    dQ = torch.zeros_like(Q)
    dK = torch.zeros_like(K)
    dV = torch.zeros_like(V)

    D = torch.sum(dO * O, dim=-1)       # (batch, num_heads, seq_len)

    # compute dK and dV

    for b in range(batch_size):
        for h in range(num_heads):

            for kv_start in range(0, seq_len, BLOCK_SIZE_KV):

                kv_end = min(kv_start + BLOCK_SIZE_KV, seq_len)
                kv_indices = torch.arange(kv_start, kv_end, device=Q.device)

                K_block = K[b, h, kv_start:kv_end, :]
                V_block = V[b, h, kv_start:kv_end, :]

                dK_block = torch.zeros((kv_end - kv_start, head_dim), device=Q.device, dtype=torch.float32)
                dV_block = torch.zeros((kv_end - kv_start, head_dim), device=Q.device, dtype=torch.float32)

                if is_causal:
                    q_ranges = [(kv_start, kv_end, "diagonal"), (kv_end, seq_len, "postdiagonal")]
                else:
                    q_ranges = [(0, seq_len, "full")]


                for q_start_range, q_end_range, pass_type in q_ranges:

                    for q_start in range(q_start_range, q_end_range, BLOCK_SIZE_Q):

                        # make sure to only index upto the Q that we care about!
                        q_end = min(q_start + BLOCK_SIZE_Q, q_end_range)

                        # store upto the Q indices we are processing right now
                        q_indices = torch.arange(q_start, q_end, device=Q.device)

                        # load the Q block and corresponding gradients and stats
                        Q_block = Q[b, h, q_start:q_end, :]         # [block_q, head_dim]
                        dO_block = dO[b, h, q_start:q_end, :]       # [block_q, head_dim]
                        M_block = M[b, h, q_start:q_end]            # [block_q]
                        D_block = D[b, h, q_start:q_end]            # [block_q]


                        S_T_block = K_block @ Q_block.T     # block_kv @ block_q

                        P_T_block = torch.exp2(S_T_block - M_block[None, :])    # block_kv x block_q

                        if pass_type == "diagonal":
                            causal_mask = kv_indices[:, None] <= q_indices[None, :]     # [block_kv, block_q]

                            # mask fill the block
                            P_T_block = P_T_block.masked_fill(~causal_mask, 0.0)


                        dV_block += P_T_block @ dO_block

                        dP_T_block = V_block @ dO_block.T

                        dS_T_block = P_T_block * (dP_T_block - D_block[None, :])

                        dS_T_block = dS_T_block * LN2

                        dK_block += dS_T_block @ Q_block

                dK[b, h, kv_start:kv_end] = dK_block
                dV[b, h, kv_start:kv_end] = dV_block


            # compute dQ
            for q_start in range(0, seq_len, BLOCK_SIZE_Q):

                # make sure we only grab up to the valid Q
                q_end = min(q_start + BLOCK_SIZE_Q, seq_len)

                # store indexes of Q being processed
                q_indices = torch.arange(q_start, q_end, device=Q.device)

                # load block of queries and corresponding data
                Q_block = Q[b, h, q_start:q_end, :]     # [block_q, head_dim]
                dO_block = dO[b, h, q_start:q_end, :]   # [block_q, head_dim]
                M_block = M[b, h, q_start:q_end]        # [block_q]
                D_block = D[b, h, q_start:q_end]        # [block_q]

                # initialize accumulator for this block
                dQ_block = torch.zeros((q_end - q_start, head_dim), device=Q.device, dtype=torch.float32)

                if is_causal:
                    # for causal attention
                    # 1. process all blocks BEFORE the diagonal (keys < queries)
                    # 2. process the diagonal block (where queries == keys in block range)

                    # pre-diagonal blocks: keys from 0 to q_start
                    kv_ranges_pre_diag = [(0, q_start, "pre_diagonal")]

                    # diagonal block: keys from q_start to q_end
                    kv_ranges_diagonal = [(q_start, q_end, "diagonal")]

                    kv_ranges = kv_ranges_pre_diag + kv_ranges_diagonal

                else:
                    # for non-causal: process all keys/values
                    kv_ranges = [(0, seq_len, "full")]



                for kv_start_range, kv_end_range, pass_type in kv_ranges:

                    # process keys/values in blocks within this range
                    for kv_start in range(kv_start_range, kv_end_range, BLOCK_SIZE_KV):

                        # make sure to only index up to the K/V that we care about!
                        kv_end = min(kv_start + BLOCK_SIZE_KV, kv_end_range)

                        # store the K/V indices we are processing right now
                        kv_indices = torch.arange(kv_start, kv_end, device=Q.device)

                        # load K and V blocks
                        K_block = K[b, h, kv_start:kv_end, :]       # [block_kv, head_dim]
                        V_block = V[b, h, kv_start:kv_end, :]       # [block_kv, head_dim]

                        S_block = Q_block @ K_block.T

                        P_block = torch.exp2(S_block - M_block[:, None])

                        if pass_type == "diagonal":
                            # for the diagonal block, mask upper triangle
                            # (queries should not attend to future keys)
                            causal_mask = q_indices[:, None] >= kv_indices[None, :]
                            P_block = P_block.masked_fill(~causal_mask, 0.0)

                        dP_block = dO_block @ V_block.T

                        dS_block = P_block * (dP_block - D_block[:, None])

                        dS_block = dS_block * LN2

                        dQ_block += dS_block @ K_block

                    dQ_block = dQ_block * softmax_scale

                    dQ[b, h, q_start:q_end, :] = dQ_block

    return dQ, dK, dV


In [ ]:
if __name__ == "__main__":
    q = torch.randn(2, 4, 32, 64, device="cuda", requires_grad=True)
    k = torch.randn(2, 4, 32, 64, device="cuda", requires_grad=True)
    v = torch.randn(2, 4, 32, 64, device="cuda", requires_grad=True)
    dO = torch.randn(2, 4, 32, 64, device="cuda", requires_grad=True)

    # pseudo forward pass
    O, M = flash_attention_forward(q, k, v, is_causal=True)

    # pseudo backward pass
    dQ, dK, dV = flash_attention_backward(q, k, v, O, dO, M, is_causal=True)

    # torch SDPA forward
    output = torch.nn.functional.scaled_dot_product_attention(
        q, k, v, is_causal=True
    )

    print("max forward pass output diff:", torch.max(torch.abs(output - O).detach().item()))

    output.backward(dO)
    print("max backward pass dQ diff:", torch.max(torch.abs(q.grad - dQ).detach().item()))
    print("max backward pass dK diff:", torch.max(torch.abs(k.grad - dK).detach().item()))
    print("max backward pass dV diff:", torch.max(torch.abs(v.grad - dV).detach().item()))